In [1]:
import os
from net import RobertaClassification
from YelpData import YelpDataset
import torch
from torch.utils.data import DataLoader
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from transformers.optimization import get_scheduler
from transformers import RobertaTokenizer,RobertaConfig,RobertaModel
from common import constants
import json
from sklearn import metrics
import logging
from tensorboardX import SummaryWriter

In [7]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 50
BATCH_SIZE = 10
MODEL_PATH = constants.MODEL_PATH
DATAFILE_PATH = constants.CLEAN_DATA_PATH

train_dataset = YelpDataset(DATAFILE_PATH,MODEL_PATH,'train_small')
test_dataset = YelpDataset(DATAFILE_PATH,MODEL_PATH,'test_small')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,drop_last=True,collate_fn=train_dataset.load_data)
test_loader =  DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,drop_last=True,collate_fn=test_dataset.load_data)

In [8]:
# Get labels name from dataset_info from dataset
file_path = './data/yelp_review_full/train/dataset_info.json'
with open(file_path, 'r') as f:
    data_info = json.load(f)

# Create directory to save model training paramaters
if not os.path.exists('params'):
    os.makedirs('params')
LABELS = data_info['features']['label']['names']

# Load Config from pretrained model
config = RobertaConfig.from_pretrained(MODEL_PATH)
num_labels = len(LABELS)
# Change the max_postion_embedding to 1024, and put the model to DEVICE
# config.max_position_embeddings = 512

# Initialize model
model = RobertaClassification(config, num_labels).to(DEVICE)

In [9]:
optimizer = torch.optim.AdamW(model.parameters(),lr = 2e-6)
loss_func = torch.nn.CrossEntropyLoss()

In [11]:
print(num_labels)

5


In [5]:
for i, (input_ids,attention_masks,labels) in enumerate (train_loader):
    input_ids,attention_masks,labels = input_ids.to(DEVICE),attention_masks.to(DEVICE), labels.to(DEVICE)
    # print(labels)
    out = model(input_ids,attention_masks)
    # print(out)
    loss = loss_func(out, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


    out = out.argmax(dim=1)
    acc = (out==labels).sum().item()/len(labels)
    print(f'loss: {loss}, acc: {acc}')

    if i > 4:
        break

loss: 2.039785861968994, acc: 0.1
loss: 1.7127870321273804, acc: 0.3
loss: 1.6709442138671875, acc: 0.2
loss: 1.6431591510772705, acc: 0.3
loss: 1.5890297889709473, acc: 0.2
loss: 1.5301014184951782, acc: 0.4
